# 26.9.14

In [112]:
import torch
print(torch.__version__)              # 2.7.1+cu128
print(torch.cuda.is_available())      # True 나와야 GPU
print(torch.cuda.get_device_name(0))  # RTX 5060

2.7.1+cu128
True
NVIDIA GeForce RTX 5060 Laptop GPU


In [113]:
print(torch.tensor([1,2,3]))              # 1차원 (OK)
print(torch.Tensor([[1,2,3],[4,5,6]]))    # 2차원 (감쌈)
print(torch.LongTensor([1,2,3]))          # OK
print(torch.FloatTensor([1,2,3]))  

tensor([1, 2, 3])
tensor([[1., 2., 3.],
        [4., 5., 6.]])
tensor([1, 2, 3])
tensor([1., 2., 3.])


In [114]:
tensor = torch.rand(1,2)
print(tensor.shape)
print(tensor.dtype)
print(tensor.device)

torch.Size([1, 2])
torch.float32
cpu


In [115]:
tensor = tensor.reshape(2, 1)
print(tensor)
print(tensor.shape)

tensor([[0.2106],
        [0.7975]])
torch.Size([2, 1])


In [116]:
tensor = torch.rand((3,3), dtype = torch.float)
print(tensor)

tensor([[0.3995, 0.1537, 0.6445],
        [0.1993, 0.1364, 0.3776],
        [0.1017, 0.7410, 0.7691]])


In [117]:
device = "cuda" if torch.cuda.is_available() else 'cpu'   # cuda 추가 + 철자
print(device)

cuda


In [118]:
cpu = torch.FloatTensor([1, 2, 3])
gpu = torch.cuda.FloatTensor([1, 2, 3])
tensor = torch.rand((1, 1), device = device)
print(cpu)
print(gpu)
print(tensor)

tensor([1., 2., 3.])
tensor([1., 2., 3.], device='cuda:0')
tensor([[0.1670]], device='cuda:0')


In [119]:
cpu = torch.FloatTensor([1, 2, 3])
gpu = cpu.cuda()
gpu2cpu = gpu.cpu()
cpu2gpu = cpu.to(device)

In [120]:
import numpy as np

ndarray = np.array([1, 2, 3], dtype = np.uint8)
print(torch.tensor(ndarray))
print(torch.Tensor(ndarray))
print(torch.from_numpy(ndarray))

tensor([1, 2, 3], dtype=torch.uint8)
tensor([1., 2., 3.])
tensor([1, 2, 3], dtype=torch.uint8)


In [121]:
tensor = torch.cuda.FloatTensor([1,2,3])
ndarray = tensor.detach().cpu().numpy()
print(ndarray)
print(type(ndarray))

[1. 2. 3.]
<class 'numpy.ndarray'>


# 비선형회귀 만들기(torch)

In [122]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from  torch.utils.data import Dataset, DataLoader, random_split

In [123]:
class CustomDataset(Dataset):
    def __init__(self, file_path):
        df = pd.read_csv(file_path)
        self.x = df.iloc[:, 0].values
        self.y = df.iloc[:,1].values
        self.length = len(df)
    
    def __getitem__(self,index):
        x = torch.FloatTensor([self.x[index] ** 2, self.x[index]])
        y = torch.FloatTensor([self.y[index]])
        return x, y
    
    def __len__(self):
        return self.length

In [124]:
class CustomModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(2, 1)

    def forward(self, x):
        x = self.layer(x)
        return x

In [125]:
dataset = CustomDataset("./non_linear.csv")
dataset_size = len(dataset)
train_size = int(dataset_size * 0.8)
val_size = int(dataset_size * 0.1)
test_size = dataset_size - train_size - val_size 

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

train_dataloader = DataLoader(
    train_dataset, 
    batch_size = 16, 
    shuffle = True, 
    drop_last = True
    )

val_dataloader = DataLoader(
    val_dataset, 
    batch_size = 4, 
    shuffle = True, 
    drop_last = True
    )

test_dataloader = DataLoader(
    test_dataset, 
    batch_size = 4, 
    shuffle = False, 
    drop_last = True
    )    

In [126]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CustomModel().to(device)
criterion = nn.MSELoss().to(device)
optimizer = optim.SGD(model.parameters(), lr = 0.0001)

In [127]:
import os
os.makedirs('./models', exist_ok=True)

checkpoint = 1
for epoch in range(1000):
    cost = 0.0
    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        cost += loss.item()
    cost = cost / len(train_dataloader)
    if (epoch + 1) % 100 == 0:
        print(epoch + 1, cost)
        torch.save(model.state_dict(), f'./models/checkpoint-{checkpoint}.pt')
        checkpoint += 1

100 0.08348997309803963
200 0.08241514191031456
300 0.08077869266271591
400 0.08057643845677376
500 0.07874791100621223
600 0.07801500484347343
700 0.07647759914398193
800 0.0769326377660036
900 0.0773736484348774
1000 0.07639143727719784


In [128]:
with torch.no_grad():
    model.eval()
    for x, y in val_dataloader:
        x = x.to(device)
        y = y.to(device)
        outputs = model(x)
        print(outputs)


tensor([[  6.2065],
        [230.7358],
        [  8.5306],
        [  7.0890]], device='cuda:0')
tensor([[ 44.4828],
        [140.4141],
        [296.2569],
        [151.0142]], device='cuda:0')
tensor([[ 26.8493],
        [  2.4603],
        [  0.3695],
        [128.1967]], device='cuda:0')
tensor([[  0.7874],
        [302.3423],
        [ 17.8193],
        [ 50.5256]], device='cuda:0')
tensor([[  3.2956],
        [220.1753],
        [238.8101],
        [ 49.2829]], device='cuda:0')


In [129]:
import os
os.makedirs('./models', exist_ok=True)
torch.save(model.state_dict(), './models/model.pt')

In [130]:
torch.save(model.state_dict(), './models/model_state_dict.pt')

In [131]:
model = CustomModel().to(device)
model.load_state_dict(torch.load('./models/model.pt', map_location = device))

<All keys matched successfully>

In [132]:
print(model)

CustomModel(
  (layer): Linear(in_features=2, out_features=1, bias=True)
)


In [133]:
model = CustomModel().to(device)
model_state_dict = torch.load('./models/model_state_dict.pt', map_location = device)
model.load_state_dict(model_state_dict)

<All keys matched successfully>

In [134]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

In [135]:
class CustomDataset(Dataset):
    def __init__(self, file_path):
        df = pd.read_csv(file_path)
        self.x1 = df.iloc[:, 0].values
        self.x2 = df.iloc[:, 1].values
        self.x3 = df.iloc[:, 2].values
        self.y = df.iloc[:, 3].values
        self.length = len(df)

    def __getitem__(self, index):
        x = torch.FloatTensor([self.x1[index], self.x2[index], self.x3[index]])
        y = torch.FloatTensor([int(self.y[index])])
        return x, y
    
    def __len__(self):
        return self.length

In [136]:
class CustomModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(3, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.layer(x)
        return x

In [137]:
dataset = CustomDataset('./binary.csv')
dataset_size = len(dataset)
train_size = int(dataset_size * 0.8)
val_size = int(dataset_size * 0.1)
test_size = dataset_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size], generator = torch.Generator().manual_seed(42)
    )

train_dataloader = DataLoader(
    train_dataset, 
    batch_size = 64, 
    shuffle = True, 
    drop_last = True
    )

val_dataloader = DataLoader(
    val_dataset, 
    batch_size = 4, 
    shuffle = True, 
    drop_last = True
    )

test_dataloader = DataLoader(
    test_dataset, 
    batch_size = 4, 
    shuffle = False, 
    drop_last = True
    )    